In [821]:
import re

In [822]:
with open("corpus_RNN_Médine.txt", "r", encoding="utf-8", errors="replace") as f:
    corpus = f.read()

### I will need to regroup each song and then count the number of occurences from each of my parts

Ok some of the songs seems empty in parts but can contain's some text 

In [823]:
all_parts = [(m.group(1), m.start(), m.end()) for m in re.finditer(r"<(\w+)>", corpus)]

In [824]:
for i in range(len(all_parts)-3) :
    nn = len(all_parts)-3 -i
    if all_parts[nn-2][0] == "BEGINNING" :
        if all_parts[nn-1][0] == "END_SECTION" :
            if all_parts[nn][0] == "END" :
                if all_parts[nn][1] - all_parts[nn-2][1] > 1000 :
                    print(all_parts[nn-2],all_parts[nn])

('BEGINNING', 586601, 586612) ('END', 588617, 588622)
('BEGINNING', 580667, 580678) ('END', 585047, 585052)
('BEGINNING', 576988, 576999) ('END', 578999, 579004)
('BEGINNING', 531888, 531899) ('END', 536717, 536722)
('BEGINNING', 525875, 525886) ('END', 527470, 527475)
('BEGINNING', 508298, 508309) ('END', 512944, 512949)
('BEGINNING', 312622, 312633) ('END', 314461, 314466)
('BEGINNING', 309755, 309766) ('END', 312615, 312620)
('BEGINNING', 224582, 224593) ('END', 228242, 228247)
('BEGINNING', 203590, 203601) ('END', 205928, 205933)
('BEGINNING', 182407, 182418) ('END', 188308, 188313)
('BEGINNING', 84547, 84558) ('END', 89278, 89283)


In [825]:
for i in range(len(all_parts)-3) :
    nn = len(all_parts)-3 -i
    if all_parts[nn-2][0] == "BEGINNING" :
        if all_parts[nn-1][0] == "END_SECTION" :
            if all_parts[nn][0] == "END" :
                if all_parts[nn][1] - all_parts[nn-2][1] < 1000 :
                    print(all_parts[nn-2],all_parts[nn])
                    part_1_corp = corpus[:all_parts[nn-2][1]]
                    part_2_corp = corpus[all_parts[nn][2]:]
                    corpus = part_1_corp+part_2_corp

('BEGINNING', 599709, 599720) ('END', 600614, 600619)
('BEGINNING', 589815, 589826) ('END', 590739, 590744)
('BEGINNING', 579006, 579017) ('END', 579734, 579739)
('BEGINNING', 477411, 477422) ('END', 478098, 478103)


In [826]:
corpus_markov = corpus

In [827]:
all_parts = [(m.group(1), m.start(), m.end()) for m in re.finditer(r"<(\w+)>", corpus_markov)]

count = 0
beg = 0
endi =0

for i in all_parts :
    if i[0] == "BEGINNING" :
        count = 0
        beg = i[1]
    else : 
        if i[0] == "END" :
            count+=1
            endi = i[2]
            if count >=100 :
                print(count, beg, endi)
                break
        else : 
            count+=1

corpus_markov = corpus_markov[:beg] + corpus_markov[endi:]

321 449910 454399


A song has 321 parts in it, after inspection it's not an error from my code : http://genius.com/Artistes-divers-morts-pour-rien-lyrics

It's an outlier that can be explained by the fact that it's related to a specific event that happened in 2005

### Featuring will be removed from the Markov Chain as this model will only generate the lyrics of one artist, so the structure of the songs can't be of a featuring.

In [828]:
print("Nb of songs :",len(re.findall(r"<END>", corpus_markov)), "Nb of featurings : ",len(re.findall(r"<BEGINNING>True.*?<END>", corpus_markov, flags=re.DOTALL)))

Nb of songs : 240 Nb of featurings :  96


In [829]:
corpus_solo = re.sub(r"<BEGINNING>True.*?<END>\n\n", " ", corpus_markov, flags=re.DOTALL)

In [830]:
all_parts = [(m.group(1), m.start(), m.end()) for m in re.finditer(r"<(\w+)>", corpus_solo)]

### We need to delete the songs where only one part has been identified

Most of the time that can be explained by an error in collecting the parts because the format can changed during the year of release

In [831]:
all_parts = [(m.group(1), m.start(), m.end()) for m in re.finditer(r"<(\w+)>", corpus_solo)]

for i in range(len(all_parts)-3) :
    nn = len(all_parts)-3 -i
    if all_parts[nn-2][0] == "BEGINNING" :
        if all_parts[nn-1][0] == "END_SECTION" :
            if all_parts[nn][0] == "END" :
                print(all_parts[nn-2], all_parts[nn])
                part_1_corp = corpus_solo[:all_parts[nn-2][1]]
                part_2_corp = corpus_solo[all_parts[nn][2]:]
                corpus_solo = part_1_corp+part_2_corp

('BEGINNING', 441126, 441137) ('END', 443142, 443147)
('BEGINNING', 435192, 435203) ('END', 439572, 439577)
('BEGINNING', 433170, 433181) ('END', 435181, 435186)
('BEGINNING', 413194, 413205) ('END', 418023, 418028)
('BEGINNING', 411590, 411601) ('END', 413185, 413190)
('BEGINNING', 398617, 398628) ('END', 403263, 403268)
('BEGINNING', 254061, 254072) ('END', 255900, 255905)
('BEGINNING', 251194, 251205) ('END', 254054, 254059)
('BEGINNING', 182827, 182838) ('END', 186487, 186492)
('BEGINNING', 163374, 163385) ('END', 165712, 165717)
('BEGINNING', 142191, 142202) ('END', 148092, 148097)
('BEGINNING', 58576, 58587) ('END', 63307, 63312)


Now we will need to annotate the parts by the number of times they appeared, this will prevent that the Markov chain learn that a song can have like 7 REFRAIN.

This will also help in finding errors in the lyrics or structure in the corpus

In [ ]:
parts = re.findall(r"<(\w+)>", corpus_solo) #Identify each part
regrouped = " ".join(parts) #re.findall gave back a list, we need a full text to perform next modif
regrouped = re.sub("END_SECTION ", "", regrouped) 
splitted = regrouped.split("END")

splitted = [i.strip() for i in splitted[:-1]]

In [ ]:
def add_cumsum_tags(parts):
    counts = {}
    result = []
    for p in parts:
        counts[p] = counts.get(p, 0)
        if counts[p] != 0 :
            result.append(f"{p}_{counts[p]}")
        else : 
            result.append(f"{p}")
        counts[p] += 1
    return result

def transform_to_num_structure(splitted) :
    #This function will serve only if I decide to aggregate more songs from other rapper because we need to have a sufficient nb of sample
    #or we will have a huge nb of single interaction such as couplet_3 refrain_3 ...
    new_splitted = []
    
    for split in splitted :
        interm = add_cumsum_tags(split.split(" "))
        new_splitted.append(interm)

    count_all = {}
        
    for elem in new_splitted :
        for length in range(len(elem)-1) :
            name = elem[length]+" "+elem[length+1]
            interm = count_all.get(name,0)+1
            count_all[name] = interm
    
    return new_splitted, count_all

In [862]:
new_split, count_all = transform_to_num_structure(splitted)

Maybe I will collect data structure of multiple artists and then aggregate to get a better Markov Chain but if I do this, it will be in the future not now

good_parts = re.sub("END_SECTION ", ""," ".join(all_parts)).split(" ")

In [863]:
count_all

{'BEGINNING REFRAIN': 15,
 'REFRAIN COUPLET': 15,
 'COUPLET REFRAIN_1': 16,
 'REFRAIN_1 COUPLET_1': 13,
 'COUPLET_1 OUTRO': 7,
 'OUTRO END': 65,
 'BEGINNING COUPLET': 69,
 'COUPLET REFRAIN': 58,
 'REFRAIN COUPLET_1': 59,
 'COUPLET_1 REFRAIN_1': 59,
 'REFRAIN_1 OUTRO': 20,
 'BEGINNING INTRO': 46,
 'INTRO COUPLET': 40,
 'COUPLET OUTRO': 11,
 'COUPLET PONT': 18,
 'PONT REFRAIN': 8,
 'COUPLET_1 PONT_1': 7,
 'PONT_1 REFRAIN_1': 3,
 'REFRAIN_1 COUPLET_2': 23,
 'COUPLET_2 REFRAIN_2': 16,
 'REFRAIN_2 END': 15,
 'PONT COUPLET_1': 11,
 'COUPLET_1 END': 4,
 'INTRO REFRAIN': 5,
 'BEGINNING OUTRO': 2,
 'COUPLET COUPLET_1': 13,
 'REFRAIN OUTRO': 1,
 'REFRAIN PONT': 4,
 'REFRAIN_1 END': 20,
 'COUPLET_1 REFRAIN_2': 12,
 'REFRAIN_2 COUPLET_3': 3,
 'COUPLET_3 END': 1,
 'COUPLET_3 REFRAIN_3': 2,
 'REFRAIN_3 END': 3,
 'REFRAIN_2 OUTRO': 12,
 'REFRAIN_1 PONT_1': 3,
 'PONT_1 REFRAIN_2': 2,
 'PONT COUPLET': 2,
 'PONT_1 COUPLET_1': 1,
 'REFRAIN_2 COUPLET_2': 2,
 'COUPLET_2 END': 10,
 'COUPLET_1 COUPLET_2': 9,

In [864]:
def count_each_interact(splitted) :
    
    for i in range(len(splitted)) :
        splitted[i] = splitted[i]+" END"

    good_parts = (" ".join(splitted)).split(" ")

    dict_glob = {}
    for i in range(len(good_parts)-1) :
        try : 
            dict_glob[f"{good_parts[i]} {good_parts[i+1]}"] += 1
        except :
            dict_glob[f"{good_parts[i]} {good_parts[i+1]}"] = 1

    beginn = {}
    intro = {}
    couplet = {}
    pont = {}
    refrain = {}
    outro = {}

    order = ["BEGINNING","INTRO", "COUPLET", "PONT", "REFRAIN", "OUTRO", "END"]
    list_dicti = [beginn, intro, couplet, pont, refrain, outro]
    for part in list_dicti :
        for j in order :
            part[j] = 0    
    
    for i in dict_glob.keys() :
        actual = i.split(" ")[0]
        next = i.split(" ")[1]

        if actual == "BEGINNING" :  
            beginn[next] = dict_glob[i]

        elif actual == "INTRO" :
            intro[next] = dict_glob[i]

        elif actual == "COUPLET" :
            couplet[next] = dict_glob[i]
        elif actual == "PONT" :
            pont[next] = dict_glob[i]
        elif actual == "REFRAIN" :
            refrain[next] = dict_glob[i]
        elif actual == "OUTRO" :
            outro[next] = dict_glob[i]

    return list_dicti

In [866]:
all_count = count_each_interact(splitted)

## Some errors can be found in the differents parts, we will remove what we think is impossible

In [867]:
all_count[0]

{'BEGINNING': 0,
 'INTRO': 46,
 'COUPLET': 69,
 'PONT': 0,
 'REFRAIN': 15,
 'OUTRO': 2,
 'END': 0}

In [868]:
all_parts = [(m.group(1), m.start(), m.end()) for m in re.finditer(r"<(\w+)>", corpus_solo)]

for i in range(len(all_parts)-4) :
    nn = len(all_parts)-4 -i
    if all_parts[nn-3][0] == "BEGINNING" :
            if all_parts[nn-2][0] == "OUTRO" :
                if all_parts[nn-1][0] == "END_SECTION" :
                     if all_parts[nn][0] == "END" :
                        part_1_corp = corpus_solo[:all_parts[nn-3][1]]
                        part_2_corp = corpus_solo[all_parts[nn][2]:]
                        corpus_solo = part_1_corp+part_2_corp

We consider that it's not possible to have the end of the music after the beginning

Two of the songs have only detect the beginning because the format that indicates the part are different, we discard them for calculating the probability of change of state

In [869]:
parts = re.findall(r"<(\w+)>", corpus_solo) #Identify each part
regrouped = " ".join(parts) #re.findall gave back a list, we need a full text to perform next modif
regrouped = re.sub("END_SECTION ", "", regrouped) 
splitted = regrouped.split("END")

splitted = [i.strip() for i in splitted[:-1]]

In [870]:
all_count = count_each_interact(splitted)

In [872]:
all_count[0] #begin

{'BEGINNING': 0,
 'INTRO': 46,
 'COUPLET': 69,
 'PONT': 0,
 'REFRAIN': 15,
 'OUTRO': 0,
 'END': 0}

In [873]:
all_count[1] #intro

{'BEGINNING': 0,
 'INTRO': 0,
 'COUPLET': 41,
 'PONT': 1,
 'REFRAIN': 5,
 'OUTRO': 0,
 'END': 0}

In [874]:
all_count[2] #couplet

{'BEGINNING': 0,
 'INTRO': 0,
 'COUPLET': 24,
 'PONT': 27,
 'REFRAIN': 163,
 'OUTRO': 26,
 'END': 26}

In [875]:
all_count[3] #pont

{'BEGINNING': 0,
 'INTRO': 0,
 'COUPLET': 17,
 'PONT': 0,
 'REFRAIN': 16,
 'OUTRO': 4,
 'END': 0}

In [876]:
all_count[4] #refrain

{'BEGINNING': 0,
 'INTRO': 1,
 'COUPLET': 115,
 'PONT': 9,
 'REFRAIN': 5,
 'OUTRO': 33,
 'END': 41}

In [886]:
all_count[4]["REFRAIN"] = 0

Intro isn't an error because I found songs where there was two intro and one for each rapper

In [887]:
all_count[5] #outro

{'BEGINNING': 0,
 'INTRO': 0,
 'COUPLET': 0,
 'PONT': 0,
 'REFRAIN': 0,
 'OUTRO': 0,
 'END': 63}

## Now we need to obtain the proba for each state to go to another

In [888]:
matrix = []
for i in all_count :
    inter_matrix = []
    for j in i :
        inter_matrix.append(i[j]/sum(i.values()))
    matrix.append(inter_matrix)

In [889]:
order = ["BEGINNING","INTRO", "COUPLET", "PONT", "REFRAIN", "OUTRO", "END"]

In [890]:
print(order[:])
print(matrix)

['BEGINNING', 'INTRO', 'COUPLET', 'PONT', 'REFRAIN', 'OUTRO', 'END']
[[0.0, 0.35384615384615387, 0.5307692307692308, 0.0, 0.11538461538461539, 0.0, 0.0], [0.0, 0.0, 0.8723404255319149, 0.02127659574468085, 0.10638297872340426, 0.0, 0.0], [0.0, 0.0, 0.09022556390977443, 0.10150375939849623, 0.6127819548872181, 0.09774436090225563, 0.09774436090225563], [0.0, 0.0, 0.4594594594594595, 0.0, 0.43243243243243246, 0.10810810810810811, 0.0], [0.0, 0.005025125628140704, 0.5778894472361809, 0.04522613065326633, 0.0, 0.1658291457286432, 0.20603015075376885], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]]


In [902]:
def generate_a_song_structure(matrix) :
    
    song_struct = [0]

    index = [i for i, p in enumerate(matrix[0]) if p>0]
    proba = [p for p in matrix[0] if p > 0]

    cumsum = np.cumsum(proba)
    r = np.random.rand()

    idx = np.searchsorted(cumsum, r)
    selected_value = index[idx] 
    song_struct.append(selected_value)

    end = False

    while end == False :

        index = [i for i, p in enumerate(matrix[selected_value]) if p>0]
        proba = [p for p in matrix[selected_value] if p > 0]

        cumsum = np.cumsum(proba)
        r = np.random.rand()

        idx = np.searchsorted(cumsum, r)
        selected_value = index[idx]
        song_struct.append(selected_value)

        if selected_value == 6 :
            end = True

    print([order[i] for i in song_struct])

In [903]:
for run in range(10) :
    generate_a_song_structure(matrix)

['BEGINNING', 'REFRAIN', 'COUPLET', 'REFRAIN', 'OUTRO', 'END']
['BEGINNING', 'COUPLET', 'REFRAIN', 'COUPLET', 'REFRAIN', 'OUTRO', 'END']
['BEGINNING', 'COUPLET', 'REFRAIN', 'COUPLET', 'END']
['BEGINNING', 'REFRAIN', 'COUPLET', 'END']
['BEGINNING', 'INTRO', 'COUPLET', 'COUPLET', 'REFRAIN', 'COUPLET', 'END']
['BEGINNING', 'INTRO', 'COUPLET', 'REFRAIN', 'COUPLET', 'REFRAIN', 'COUPLET', 'OUTRO', 'END']
['BEGINNING', 'COUPLET', 'PONT', 'REFRAIN', 'END']
['BEGINNING', 'INTRO', 'COUPLET', 'REFRAIN', 'END']
['BEGINNING', 'COUPLET', 'REFRAIN', 'OUTRO', 'END']
['BEGINNING', 'COUPLET', 'OUTRO', 'END']


This seems ok for now. 

So the matrix will be exported and used during the model generation of text